# 05 — Regenerate the MT data from clean text

The previous MT files came from the Excel export, which carried a
UTF-8/latin-1 corruption affecting 13.8% of sentences. Both translation systems
were therefore trained and evaluated on damaged Mizo source text.

This notebook rebuilds all three MT files directly from the deduplicated corpus
and injects entity markup from character offsets rather than string replacement.

**Run from the repository root.** Kernel: `Python (tka)`. A few minutes.

## Cell 1: Setup

In [1]:
from pathlib import Path
import json, re, sys, random
from collections import Counter

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print(f"Repo root: {ROOT}")

PROC   = ROOT / "data" / "processed"
CORPUS = PROC / "mizo_ner_annotations_dedup_aggressive.jsonl"
OUT    = PROC / "mt_v2"
OUT.mkdir(exist_ok=True)
RES    = ROOT / "results" / "mt"
RES.mkdir(parents=True, exist_ok=True)

print(("  ok   " if CORPUS.exists() else "  MISS ") + str(CORPUS.relative_to(ROOT)))
if not CORPUS.exists():
    sys.exit("Corpus not found")

OLD = PROC          # the previous flat MT files live here

Repo root: C:\Users\Haulai\mizo-ner
  ok   data\processed\mizo_ner_annotations_dedup_aggressive.jsonl


## Cell 2: Load the corpus

In [2]:
records = []
with open(CORPUS, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Sentences : {len(records):,}")
print(f"Entities  : {sum(len(r['entities']) for r in records):,}")
print(f"Keys      : {list(records[0].keys())}")

missing_en = sum(1 for r in records if not r.get("english_source", "").strip())
print(f"Missing English side: {missing_en:,}")

Sentences : 441,178
Entities  : 590,655
Keys      : ['text', 'entities', 'english_source']
Missing English side: 0


## Cell 3: Markup injection

Spans are wrapped in guillemet delimiters. Injection walks right to left so
earlier offsets stay valid, and overlapping spans resolve to the longest. The
annotation marks the stem, so an inflectional suffix falls outside the closing
delimiter: `Liana-an` becomes `«PERSON» Liana «/PERSON»-an`.

In [3]:
OPEN, CLOSE = "\u00ab", "\u00bb"

def inject(text, entities):
    ents = sorted(entities, key=lambda e: (e[0], -(e[1] - e[0])))
    kept, last_end = [], -1
    for s, e, lab in ents:
        if s >= last_end:
            kept.append((s, e, lab)); last_end = e
    out = text
    for s, e, lab in sorted(kept, key=lambda x: -x[0]):
        out = (out[:s] + f"{OPEN}{lab}{CLOSE} {out[s:e]} {OPEN}/{lab}{CLOSE}" + out[e:])
    return out, len(kept)

def strip_markup(s):
    s = re.sub(OPEN + r"[A-Z_]+" + CLOSE + " ", "", s)
    s = re.sub(" " + OPEN + r"/[A-Z_]+" + CLOSE, "", s)
    return s

# self-test on known cases before touching the corpus
checks = [
    ("Kar thum chawlh Liana-an a la.", [[16,21,"PERSON"]]),
    ("Aizawlah an thuthmun tur.",      [[0,6,"GPE"]]),
    ("Mizo tawng hi \u1e6dha a ni.",     [[0,4,"NORP"]]),
]
for t, e in checks:
    o, _ = inject(t, e)
    assert strip_markup(o) == t, f"round trip failed on: {t}"
    print(f"  {o}")
print("Injector self-test passed.")

  Kar thum chawlh «PERSON» Liana «/PERSON»-an a la.
  «GPE» Aizawl «/GPE»ah an thuthmun tur.
  «NORP» Mizo «/NORP» tawng hi ṭha a ni.
Injector self-test passed.


## Cell 4: Build the three files

In [4]:
mizo_plain, mizo_tagged, english = [], [], []
total_spans = kept_spans = 0
skipped = 0

for r in records:
    text = r["text"].strip()
    en   = r.get("english_source", "").strip()
    if not text or not en:
        skipped += 1
        continue
    ents = r.get("entities", [])
    tagged, kept = inject(text, ents)
    total_spans += len(ents); kept_spans += kept
    mizo_plain.append(text)
    mizo_tagged.append(tagged)
    english.append(en)

print(f"Sentences written : {len(mizo_plain):,}")
print(f"Skipped (empty)   : {skipped:,}")
print(f"Spans available   : {total_spans:,}")
print(f"Spans injected    : {kept_spans:,}  ({kept_spans/total_spans*100:.2f}%)")
print(f"\nplain : {mizo_plain[0]}")
print(f"tagged: {mizo_tagged[0]}")
print(f"english: {english[0]}")

Sentences written : 441,178
Skipped (empty)   : 0
Spans available   : 590,655
Spans injected    : 586,706  (99.33%)

plain : Kar thum chawlh Liana-an a la.
tagged: Kar thum chawlh «PERSON» Liana «/PERSON»-an a la.
english: Liana took three weeks off.


## Cell 5: Verify the encoding is clean

The corruption signature is a sequence such as `Ã¢`, `á¹`, or `â€™` where a
Mizo diacritic or curly quote should be. Count them in the new files and, for
comparison, in the old ones.

In [5]:
MOJI = re.compile(r"Ã.|á¹.|â€.|Â.")

def moji_count(lines):
    return sum(1 for l in lines if MOJI.search(l))

new_bad = moji_count(mizo_plain)
print(f"new mizo_plain  : {new_bad:,} of {len(mizo_plain):,} lines show corruption "
      f"({new_bad/len(mizo_plain)*100:.2f}%)")

old_path = OLD / "mizo_plain.txt"
if old_path.exists():
    old_lines = open(old_path, encoding="utf-8").read().splitlines()
    old_bad = moji_count(old_lines)
    print(f"old mizo_plain  : {old_bad:,} of {len(old_lines):,} lines "
          f"({old_bad/len(old_lines)*100:.2f}%)")
    print(f"\nlines repaired : {old_bad - new_bad:,}")
else:
    print("old mizo_plain.txt not found for comparison")

# how many sentences carry Mizo-specific characters at all
DIA = re.compile(r"[\u1e6d\u1e6cāâêîôûáéíóú\u2018\u2019\u201c\u201d]")
print(f"\nsentences with diacritics or curly quotes: "
      f"{sum(1 for l in mizo_plain if DIA.search(l)):,}")

new mizo_plain  : 318 of 441,178 lines show corruption (0.07%)
old mizo_plain  : 63,316 of 441,178 lines (14.35%)

lines repaired : 62,998

sentences with diacritics or curly quotes: 62,196


## Cell 6: Split 90/5/5

A fresh split, with indices saved this time so any later experiment can
reproduce it exactly without depending on library behaviour.

In [6]:
SEED = 42
n = len(mizo_plain)
idx = list(range(n))
random.Random(SEED).shuffle(idx)

n_test = n_val = int(round(n * 0.05))
test_idx  = sorted(idx[:n_test])
val_idx   = sorted(idx[n_test:n_test + n_val])
train_idx = sorted(idx[n_test + n_val:])

print(f"train {len(train_idx):>8,}")
print(f"val   {len(val_idx):>8,}")
print(f"test  {len(test_idx):>8,}")
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
assert len(train_idx) + len(val_idx) + len(test_idx) == n
print("\nsplits are disjoint and complete")

json.dump({"seed": SEED, "n": n,
           "train": train_idx, "val": val_idx, "test": test_idx},
          open(OUT / "split_indices.json", "w"))

train  397,060
val     22,059
test    22,059

splits are disjoint and complete


## Cell 7: Write the files

In [7]:
def dump(path, lines):
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write("\n".join(lines) + "\n")

dump(OUT / "mizo_plain.txt",  mizo_plain)
dump(OUT / "mizo_tagged.txt", mizo_tagged)
dump(OUT / "english.txt",     english)

for name, ids in (("train", train_idx), ("val", val_idx), ("test", test_idx)):
    dump(OUT / f"{name}_mizo_plain.txt",  [mizo_plain[i]  for i in ids])
    dump(OUT / f"{name}_mizo_tagged.txt", [mizo_tagged[i] for i in ids])
    dump(OUT / f"{name}_english.txt",     [english[i]     for i in ids])

print(f"Written to {OUT.relative_to(ROOT)}:")
for p in sorted(OUT.iterdir()):
    print(f"  {p.name:<26}{p.stat().st_size/1024**2:>8.1f} MB")

Written to data\processed\mt_v2:
  english.txt                   25.1 MB
  mizo_plain.txt                25.2 MB
  mizo_tagged.txt               36.8 MB
  split_indices.json             3.3 MB
  test_english.txt               1.3 MB
  test_mizo_plain.txt            1.3 MB
  test_mizo_tagged.txt           1.8 MB
  train_english.txt             22.6 MB
  train_mizo_plain.txt          22.7 MB
  train_mizo_tagged.txt         33.1 MB
  val_english.txt                1.3 MB
  val_mizo_plain.txt             1.3 MB
  val_mizo_tagged.txt            1.8 MB


## Cell 8: Entity coverage of the test split

In [8]:
tagged_test = [mizo_tagged[i] for i in test_idx]
TAG = re.compile(OPEN + r"([A-Z_]+)" + CLOSE)

per_sentence_types = [set(TAG.findall(t)) for t in tagged_test]
type_sent_counts = Counter(t for s in per_sentence_types for t in s)
with_entity = sum(1 for s in per_sentence_types if s)

print(f"test sentences            : {len(tagged_test):,}")
print(f"  with at least one entity: {with_entity:,} ({with_entity/len(tagged_test)*100:.1f}%)")
print(f"  total entity mentions   : {sum(len(TAG.findall(t)) for t in tagged_test):,}")
print(f"\n{'Entity':<14}{'sentences':>11}")
print("-" * 25)
for lab, c in type_sent_counts.most_common():
    print(f"{lab:<14}{c:>11,}")
print("-" * 25)
print(f"{'sum':<14}{sum(type_sent_counts.values()):>11,}"
      "   (exceeds sentence count: a sentence may hold several types)")

json.dump({"test_sentences": len(tagged_test),
           "test_sentences_with_entity": with_entity,
           "test_sentences_by_type": dict(type_sent_counts)},
          open(RES / "mt_v2_test_composition.json", "w"), indent=2)

test sentences            : 22,059
  with at least one entity: 22,059 (100.0%)
  total entity mentions   : 29,420

Entity          sentences
-------------------------
PERSON             13,719
GPE                 5,077
ORG                 4,645
NORP                1,000
LOC                   341
LANGUAGE              226
WORK_OF_ART           201
PRODUCT               174
FAC                   166
EVENT                  31
LAW                    21
-------------------------
sum                25,601   (exceeds sentence count: a sentence may hold several types)


## Cell 9: Compare against the previous MT data

How far the corrected files differ from what the earlier experiment used.

In [9]:
old_plain = OLD / "mizo_plain.txt"
if old_plain.exists():
    old = open(old_plain, encoding="utf-8").read().splitlines()
    print(f"old lines: {len(old):,}   new lines: {len(mizo_plain):,}")
    common = min(len(old), len(mizo_plain))
    same = sum(1 for a, b in zip(old[:common], mizo_plain[:common]) if a == b)
    print(f"identical on first {common:,} lines: {same:,} ({same/common*100:.2f}%)")
    shown = 0
    for a, b in zip(old, mizo_plain):
        if a != b:
            print(f"\n  old: {a[:100]}")
            print(f"  new: {b[:100]}")
            shown += 1
            if shown >= 5:
                break
    print("\nNote: line order may differ between the two builds, so a low identity"
          "\nrate here is expected and not itself a problem.")
else:
    print("no previous mizo_plain.txt to compare against")

old lines: 441,178   new lines: 441,178
identical on first 441,178 lines: 316,564 (71.75%)

  old: Faka thusawi kha lo awih ila á¹­ha tur.
  new: Faka thusawi kha lo awih ila ṭha tur.

  old: "Tu nge Mary chu, leh engtin nge min hriat?"
  new: Tu nge Mary chu, leh engtin nge min hriat?

  old: Thawhá¹­an zan Serie A inkhel dangah Atalanta chuan Lecce chu 4-0 ngawtin an sawp bawk.
  new: Thawhṭan zan Serie A inkhel dangah Atalanta chuan Lecce chu 4-0 ngawtin an sawp bawk.

  old: Kum 1953-a VL. Siama lehkhabu ziak â€˜Mizo Historyâ€™ pawh chu ziak hmasa pawl a ni.
  new: Kum 1953-a VL. Siama lehkhabu ziak ‘Mizo History’ pawh chu ziak hmasa pawl a ni.

  old: "Mahse, kan Bibleah meuh pawh zai leh lÃ¢ma Pathian fak tur kan nihzia te a lang nasa hle a."
  new: Mahse, kan Bibleah meuh pawh zai leh lâma Pathian fak tur kan nihzia te a lang nasa hle a.

Note: line order may differ between the two builds, so a low identity
rate here is expected and not itself a problem.
